In [1]:
# Import pandas for data manipulation and TfidfVectorizer for text feature extraction
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the products dataset into a DataFrame
products = pd.read_csv("products.csv")

# Combine signal-rich text fields into a single 'text' column for processing.
# .fillna("") replaces missing values with empty strings to prevent concatenation errors.
# .str.lower() ensures the category text is uniform and case-insensitive.
products["text"] = (products["tags"].fillna("") + " "
                    + products["category"].str.lower() + " "
                    + products["description"].fillna(""))

# Initialize the TF-IDF Vectorizer.
# stop_words="english" removes common, uninformative words (e.g., 'and', 'the', 'is').
# max_features=2000 limits the vocabulary to the top 2,000 most frequent terms to manage memory and reduce noise.
vectorizer = TfidfVectorizer(stop_words="english", max_features=2000)

# Learn the vocabulary from the text and transform it into a TF-IDF sparse matrix.
tfidf = vectorizer.fit_transform(products["text"])

# Print the dimensions of the resulting matrix.
# Shape represents (number of products, number of unique vocabulary terms).
print("Matrix shape:", tfidf.shape)

Matrix shape: (600, 764)


In [5]:
# Import the cosine_similarity function, which measures the angle between vectors 
# to determine how similar they are (1.0 means identical, 0.0 means completely unrelated).
from sklearn.metrics.pairwise import cosine_similarity

# Compute the pairwise similarity score between all products based on their TF-IDF vectors.
# This generates a square matrix (e.g., 600x600) where every product is compared against every other product.
sim = cosine_similarity(tfidf)      

# Print the dimensions of the similarity matrix to confirm it is (number_of_products x number_of_products).
print(sim.shape)

# Print specific similarity scores to verify the calculations.
# sim[0, 0] compares the first product to itself, which will always result in 1.0.
# sim[0, 1] compares the first product to the second product, returning a score less than 1.0.
print(round(sim[0, 0], 2), round(sim[0, 1], 2))

(600, 600)
1.0 0.15


In [6]:
# Create a reverse lookup table (Pandas Series) to map a 'product_id' string to its numerical row index.
# This allows us to quickly find a product's position in the similarity matrix without searching the whole dataframe.
idx_of = pd.Series(products.index, index=products["product_id"])

def similar_products(product_id, n=5):
    # Find the integer row index for the requested target product
    i = idx_of[product_id]
    
    # Retrieve the row of similarity scores for this product from the 'sim' matrix.
    # enumerate() pairs each score with its product index, creating a list of tuples: (product_index, similarity_score)
    scores = list(enumerate(sim[i]))
    
    # Sort the list of tuples based on the similarity score (x[1]), in descending order (highest score first)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    # Extract the indices of the top 'n' products. 
    # 'if j != i' ensures we exclude the target product itself (since a product's similarity to itself is exactly 1.0).
    top = [j for j, s in scores if j != i][:n]
    
    # Retrieve the product details for these top indices from the original dataframe.
    # .copy() is used to create a clean, independent dataframe and avoid Pandas 'SettingWithCopy' warnings.
    out = products.iloc[top][["product_id", "name", "category", "rating"]].copy()
    
    # Append the actual similarity scores as a new column to the output, rounded to 2 decimal places for readability.
    out["similarity"] = [round(sim[i, j], 2) for j in top]
    
    return out

# Example usage: Find the top 5 products most similar to "PRD017" (Boat Rockerz 255 Earphones)
similar_products("PRD017")

,product_id,name,category,rating,similarity
500,PRD501,Realme Buds Wireless 2,Electronics,3.9,0.65
411,PRD412,Boult AirBass Z20 Earbuds,Electronics,4.2,0.29
63,PRD064,Noise Wired Earphones Lite,Electronics,3.4,0.24
481,PRD482,Mi Wired Earphones Lite,Electronics,4.1,0.24
147,PRD148,Realme Wired Earphones Max,Electronics,4.2,0.24


In [7]:
# Import numpy for efficient array and matrix operations
import numpy as np

# Load the historical orders dataset to see what customers have purchased
orders = pd.read_csv("orders.csv")

def recommend_for_customer(customer_id, n=5):
    # Find all unique product IDs previously purchased by this specific customer
    bought = orders.loc[orders["customer_id"] == customer_id, "product_id"].unique()
    
    # Convert those product IDs into their corresponding integer row indices.
    # The 'if p in idx_of.index' check prevents errors if a discontinued product is in the orders list but missing from the products table.
    bought_idx = [idx_of[p] for p in bought if p in idx_of.index]
    
    # Cold start fallback: If the customer hasn't bought anything (or we don't recognize their items),
    # recommend the overall top-rated products across the entire store.
    if not bought_idx:
        return products.nlargest(n, "rating")[["product_id", "name", "rating"]]

    # Build the user's preference profile.
    # We fetch the similarity rows for every item they bought, then average them across the columns (axis=0).
    # This creates a single consolidated vector representing what the customer likes in aggregate.
    profile = sim[bought_idx].mean(axis=0)
    
    # Penalize already purchased items by setting their score to -1.
    # This ensures we don't recommend products the customer already owns.
    profile[bought_idx] = -1          
    
    # Find the indices of the highest scoring products.
    # np.argsort() sorts ascending (lowest to highest), so we slice [::-1] to reverse it to descending order,
    # and then take the top 'n' results.
    top = np.argsort(profile)[::-1][:n]

    # Retrieve the product details for these top recommendations from the original dataframe.
    out = products.iloc[top][["product_id", "name", "category"]].copy()
    
    # Attach the calculated recommendation score for transparency, rounded to 2 decimal places.
    out["score"] = np.round(profile[top], 2)
    
    return out

# Example usage: Generate top 5 recommendations for a specific customer
# CUST042 bought Boat earphones + a Milton bottle
recommend_for_customer("CUST042")

,product_id,name,category,score
500,PRD501,Realme Buds Wireless 2,Electronics,0.33
90,PRD091,Cello Insulated Coffee Mug,Home & Kitchen,0.23
411,PRD412,Boult AirBass Z20 Earbuds,Electronics,0.14
128,PRD129,Pigeon Steel Lunch Box,Home & Kitchen,0.13
221,PRD222,Cello Steel Lunch Box 2.0,Home & Kitchen,0.13


In [8]:
# Create a temporary 'bought' column filled with 1s to act as a clear numerical flag for a purchase event.
pivot = (orders.assign(bought=1)
         
         # Reshape the long order log into a wide User-Item interaction matrix.
         # Rows become customers, columns become products, and cells hold the 'bought' value (1).
         # fill_value=0 ensures that items a customer hasn't purchased are marked as 0 instead of NaN.
         .pivot_table(index="customer_id", columns="product_id",
                      values="bought", fill_value=0))

# Transpose the matrix (.T) so products become the rows and customers become the columns.
# Calculate cosine similarity to find which products are frequently bought by the *same* customers.
# This creates a behavioral (collaborative filtering) similarity matrix, as opposed to the text-based one used earlier.
item_sim = cosine_similarity(pivot.T)

In [9]:
# The columns of our pivot table are the product_ids. 
# We need this list to map matrix indices back to actual product names.
pivot_product_ids = pivot.columns.tolist()

def recommend_behavioral(product_id, n=5):
    # Find the matrix index for the requested product
    if product_id not in pivot_product_ids:
        return "Product has no purchase history."
        
    i = pivot_product_ids.index(product_id)
    
    # Get the similarity scores for this product against all others
    scores = list(enumerate(item_sim[i]))
    
    # Sort by score descending, excluding the product itself
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top = [j for j, s in scores if j != i][:n]
    
    # Extract the actual product IDs for the top matches
    top_product_ids = [pivot_product_ids[j] for j in top]
    
    # Return the readable product details from the main products dataframe
    out = products[products["product_id"].isin(top_product_ids)][["product_id", "name", "category"]].copy()
    out["behavioral_score"] = [round(sim_score, 2) for j, sim_score in scores if j in top]
    
    return out

# Try it out:
recommend_behavioral("PRD017")

,product_id,name,category,behavioral_score
49,PRD050,Allen Solly Cotton T-Shirt,Fashion,0.17
87,PRD088,Milton Thermosteel Flask 1L,Home & Kitchen,0.15
129,PRD130,Sony Bluetooth Mouse Plus,Electronics,0.15
272,PRD273,Bloomsbury Cookbook Classic,Books,0.12
471,PRD472,Nivia Football Classic,Sports,0.12


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Calculate similarity between CUSTOMERS (User-User Collaborative Filtering)
# pivot shape is (customers x products)
user_sim = cosine_similarity(pivot)

# Create a mapping of customer_id to matrix index
user_ids = pivot.index.tolist()
user_idx = pd.Series(range(len(user_ids)), index=user_ids)
def recommend_from_similar_customers(customer_id, n=5):
    if customer_id not in user_idx:
        return "Customer has no purchase history."
        
    i = user_idx[customer_id]
    
    # Get similarity scores for this customer against all other customers
    scores = list(enumerate(user_sim[i]))
    
    # Sort by highest similarity, excluding the customer themselves
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top_user_indices = [j for j, s in scores if j != i][:5] # Get top 5 similar users
    
    # Find what products these similar users bought
    # pivot.iloc[top_user_indices] gives the rows of these top users
    # .sum(axis=0) adds up how many of these top users bought each item
    product_scores = pivot.iloc[top_user_indices].sum(axis=0)
    
    # Filter out items the target customer has ALREADY bought
    target_user_bought = pivot.iloc[i]
    product_scores = product_scores[target_user_bought == 0]
    
    # Get the top 'n' recommended product IDs
    top_product_ids = product_scores.nlargest(n).index.tolist()
    
    # Format the output
    out = products[products["product_id"].isin(top_product_ids)][["product_id", "name", "category"]].copy()
    out["suggested_by_similar_users"] = True
    
    return out

# Try it out for a specific customer
recommend_from_similar_customers("CUST042")

,product_id,name,category,suggested_by_similar_users
34,PRD035,Hawkins Steel Lunch Box Pro,Home & Kitchen,True
48,PRD049,Milton Pressure Cooker,Home & Kitchen,True
50,PRD051,Woodland Cotton Kurta 2.0,Fashion,True
162,PRD163,Realme Smart Watch Plus,Electronics,True
380,PRD381,Peter England Running Shorts Max M482,Fashion,True
